In [ ]:
import numpy as np
from scipy.io import loadmat

import matplotlib.pyplot as plt
import tf2onnx
import onnx
import tensorflow as tf
import onnxruntime as ort

In [ ]:
def read_samples(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()
        sample = []
        for line in lines:
            if len(line) < 2:
                continue
            sample.append([float(x) for x in line.split()])
    return np.array(sample)


In [ ]:
# open matlab file
summed_map = np.zeros((512, 512))

instrument = "3mi"
angle = 5
ray = 100

# for i in range(1, ray+1):
#     mat = loadmat(f'./data/{instrument}/{angle}/Ghost_{angle}_{i}.mat')
#     spst_map = mat['data']
#     summed_map += spst_map

# spst_map = summed_map / ray
spst_map = loadmat(f'./data/{instrument}/{angle}/full_{angle}.mat')['data']

In [ ]:
with open(f'./samples/{instrument}/{angle}/full_{angle}.txt', 'w') as f:
    for i in range(spst_map.shape[0]):
        for j in range(spst_map.shape[1]):
            if j == spst_map.shape[1] - 1:
                f.write(f'{spst_map[i, j]:.30f}')
            else:
                f.write(f'{spst_map[i, j]:.30f} ')
        if i < spst_map.shape[0] - 1:
            f.write('\n')

        

In [ ]:
spst_map = read_samples('./Results_preview/field_Ghost_ 5_2pc.txt')

log_sample = np.log10(spst_map)

plt.matshow(log_sample)
plt.show()

In [82]:
# Load the saved model
model = tf.keras.models.load_model('./model/autoencoder.h5', compile=False)
input_signature = [tf.TensorSpec([None, 512, 512, 1], tf.float32, name='x')]
# Convert the model to ONNX
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature)
onnx_model.graph.output[0].name = "output"
for node in onnx_model.graph.node:
    for i, output in enumerate(node.output):
        if output == "Dec_conv_6":
            node.output[i] = "output"
onnx.save_model(onnx_model, './model/autoencoder.onnx')

In [ ]:
spst_map.max()

In [ ]:
spst_map_dict = loadmat("./Results_preview_mat/map_1.mat")
spst_map = spst_map_dict[list(spst_map_dict.keys())[-1]]

ort_sess = ort.InferenceSession('model/autoencoder.onnx')

x = spst_map * (1000 / spst_map.max())
# clip x to [0, 1000]
x = np.clip(x, 0, 1000)
x = x[:,:, None]
x = tf.image.resize(x, (512,512)).numpy()
x = x.reshape(1, 512, 512, 1).astype(np.float32)

output = ort_sess.run(None, {'x': x})[0]
output = np.clip(output[0, ..., 0], 0, 1000)

output = output / output.max() * spst_map.max()
output = tf.image.resize(output[:,:,None], (2048,2048)).numpy()[0:,:,0]

In [ ]:
output.max(), spst_map.max()

In [ ]:
fig, ax = plt.subplots()

im = ax.imshow(np.log10(output))
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()